# **Supervised Learning**

## **JANGAN LUPA DI DOWNLOAD TERLEBIH DAHULU, ATAU DI SALIN TERLEBIH DAHULU SEBELUM MENJALANKAN PROGRAMNYA :)))**

## 1. Regression

### Preparation

Untuk persiapan, kita akan memanggil seluruh library yang akan kita pakai pada project kali ini

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from scipy.stats import shapiro
%matplotlib inline

Untuk latihan pada kali ini, kita akan memakai dataset bawaan dari Google Colab yaitu dataset `USA_Housing.csv`

In [ ]:
# membaca dataset ke dataframe
df = pd.read....('USA_Housing.csv')
df.head()

### Exploratory Data Analysis


In [ ]:
# melihat cuplikan informasi mengenai dataset
df.info()

In [ ]:
# statistika deskriptif mengenai dataset kita
df.describe(include='all')

In [ ]:
sns.pairplot(df)

Kita tinjau distribusi dari predicted variable kita dimana dalam hal ini adalah **PRICE**

In [ ]:
df['Price'].plot.hist(bins=25, figsize=(8,4))

In [ ]:
df['Price'].plot.density()

Kemudian kita tinjau pula korelasi antar variabel menggunakan correlation heatmap

Disini kita akan drop kolom address dan kita akan mempersiapkan dataset untuk training dengan cara memisahkan prediktor dan target variabel

In [ ]:
# Drop Address dari dataset (karena tidak numerik dan tidak kategorikal)
df = df.drop(columns=['....'])

df.head()

melihat korelasi antar fitur

In [ ]:
df....()

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(df.corr(),annot=True,linewidths=2)

In [ ]:
columns = list(df.columns)

# ambil kolom prediktor dan simpan di variable predictor
predictor = columns[:-1]

# ambil kolom target dan simpan di variable target
target = columns[-1]

In [ ]:
df[predictor]

In [ ]:
df[...]

In [ ]:
X = df[predictor]

In [ ]:
y = df[target]

Kemudian kita akan melakukan split train-test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)

print("Train size :", X_.....shape)
print("Test size :", X_....shape)

### Training Regression Model



In [ ]:
# menginisiasi object lm --> dari class LinearRegression
lm = LinearRegression()

In [ ]:
# melakukan training dengan menggunakan X_train dan y_train
lm.fit(X_train,y_train)

In [ ]:
print("intercept dari model kita: ", lm.intercept_)

In [ ]:
print("coefficient dari linear model kita: ", lm.coef_)

In [ ]:
cdf = pd.DataFrame(data=lm.coef_, index=X_train.columns, columns=['Coefficients'])
cdf

### Training Result Analysis

Coefficient, standard error, and T-statistic for each predictor

In [ ]:
n=X_train.shape[0] #jumalh data training
k=X_train.shape[1] #jumlah fitur independent (5 fitur)
dfN = n-k #dalammenghitung standard error, kita menggunakan derajat kebebasan(dfn)

train_pred=lm.predict(X_train) #prediksi model di data training
train_error = np.square(train_pred - y_train) #menghitung error kuadrat
sum_error=np.sum(train_error) #jumlah error kuadratnya

se=[0,0,0,0,0] #inisialisasi awal sebelum nilai standard error(se) dihitung
for i in range(k):
    r = (sum_error/dfN) #residual mse
    r = r/np.sum(np.square(X_train[list(X_train.columns)[i]]-X_train[list(X_train.columns)[i]].mean()))
    se[i]=np.sqrt(r) #menghitung se untuk masing2 koefisien

cdf['Standard Error']=se
cdf['t-statistic']=cdf['Coefficients']/cdf['Standard Error'] #menghitung t-statistic
cdf

* **Avg. Area Number of Bedrooms **memiliki nilai t-statistik paling kecil, yang mungkin berarti tidak terlalu signifikan dibandingkan fitur lainnya.
* **Avg. Area House Age dan Avg. Area Number of Rooms **memiliki pengaruh paling besar terhadap variabel target.

**Kesimpulan**

1. Koefisien vs. t-statistic: Apa yang Lebih Penting?
* Koefisien (Coefficient) → Mengukur efek absolut fitur terhadap target
Avg. Area House Age (165201.10) & Avg. Area Number of Rooms (119061.46) memiliki nilai koefisien yang jauh lebih besar daripada Avg. Area Income (21.60).
* Artinya, jika fitur ini berubah 1 unit, pengaruhnya terhadap harga rumah jauh lebih besar dibandingkan Avg. Area Income.

2. t-statistic → Mengukur signifikansi fitur terhadap model
* Avg. Area Income memiliki t-statistic tertinggi karena standard error-nya kecil, bukan karena dampaknya besar.
* Standard error kecil berarti ada sedikit variabilitas dalam data, sehingga prediksi untuk fitur ini lebih stabil.

In [ ]:
# Visualisasi prediktor vs target

l=list(cdf.index)
from matplotlib import gridspec
fig = plt.figure(figsize=(18, 10))
gs = gridspec.GridSpec(2,3)
#f, ax = plt.subplots(nrows=1,ncols=len(l), sharey=True)
ax0 = plt.subplot(gs[0])
ax0.scatter(df[l[0]],df['Price'])
ax0.set_title(l[0]+" vs. Price", fontdict={'fontsize':20})

ax1 = plt.subplot(gs[1])
ax1.scatter(df[l[1]],df['Price'])
ax1.set_title(l[1]+" vs. Price",fontdict={'fontsize':20})

ax2 = plt.subplot(gs[2])
ax2.scatter(df[l[2]],df['Price'])
ax2.set_title(l[2]+" vs. Price",fontdict={'fontsize':20})

ax3 = plt.subplot(gs[3])
ax3.scatter(df[l[3]],df['Price'])
ax3.set_title(l[3]+" vs. Price",fontdict={'fontsize':20})

ax4 = plt.subplot(gs[4])
ax4.scatter(df[l[4]],df['Price'])
ax4.set_title(l[4]+" vs. Price",fontdict={'fontsize':20})

### Evaluasi Model dan Uji Asumsi
Setelah melakukan training dan menghasilkan hasil evaluasi, kita perlu melakukan uji asumsi agar dapat memastikan bahwa model kita dapat diandalkan dan valid

In [ ]:
predictions = lm.predict(X_test) #uji model menggunakan data test

# menghitung residual dari hasil regresi kita
residuals = (y_test - predictions)

residuals

In [ ]:
# melakukan uji normalitas dengan shapiro wilk
shapiro_test_stat, shapiro_p_value = shapiro(residuals)
print("shapiro test stat : ", shapiro_test_stat)
print("shapiro p value :", shapiro_p_value)

In [ ]:
plt.figure(figsize=(10,7))
plt.title("Residuals vs. predicted values plot (Homoscedasticity)\n",fontsize=25)
plt.xlabel("Predicted house prices",fontsize=18)
plt.ylabel("Residuals", fontsize=18)
plt.scatter(x=predictions,y=y_test-predictions)

* Residuals yaitu selisih antara nilai aktual dan prediksi.
* Penyebaran residual terlihat cukup merata di sekitar nol, tanpa pola tertentu yang jelas.
* Tidak ada pola kipas atau pola berbentuk parabola, yang berarti homoskedastisitas cukup terpenuhi.
* tetapi ada sedikit outlier artiny "Model terlalu meremehkan atau melebihkan  harga rumah (harga asli jauh lebih tinggi dari prediksi) atau (harga asli jauh lebih rendah dari pada prediksi)."

In [ ]:
plt.figure(figsize=(10,7))
plt.title("Actual vs. predicted house prices",fontsize=25)
plt.xlabel("Actual test set house prices",fontsize=18)
plt.ylabel("Predicted house prices", fontsize=18)
plt.scatter(x=y_test,y=predictions)

Model cukup baik dalam memprediksi harga rumah secara umum walau ada beberapa outlier

hasil evaluasi dengan melihat MAE, RMSE, R^2

In [ ]:
print("MAE:", metrics.mean_absolute_error(y_test,predictions))
print("RMSE:", np.sqrt(metrics.mean_squared_error(y_test,predictions)))
print("R-squared test:", round(metrics.r2_score(y_test,predictions),3))
print("R-squared train:", round(metrics.r2_score(y_train,lm.predict(X_train)),3))

# **KESIMPULAN**
1. Rata-rata kesalahan prediksi sekitar 8.17% dari harga rumah rata-rata.
2. Model sudah cukup baik, dengan akurasi tinggi (R² sekitar 0.92) artiny model mampu menjelaskan 92% variansi dalam data uji walau terdapat kesalahan relatif kecil (MAE dan RMSE dalam batas wajar).
3. Namun, RMSE lebih tinggi dari MAE, menunjukkan adanya beberapa prediksi dengan error yang besar (outliers).